In [9]:
import asyncio
from agents import Agent, Runner, OpenAIChatCompletionsModel
from openai import AsyncOpenAI
from agents.tracing import set_tracing_disabled

# Disable tracing (optional)
set_tracing_disabled(True)

model = OpenAIChatCompletionsModel(
    model="minimax-m2.7:cloud",
    openai_client=AsyncOpenAI(
        api_key="ollama",  # required but ignored by Ollama
        base_url="http://localhost:11434/v1",
    ),
)

# BAD instructions — too vague
bad_agent = Agent(model=model, name="Bad Agent", instructions="You are helpful.")

# GOOD instructions — specific, structured, actionable
good_agent = Agent(
    model=model,
    name="Code Reviewer",
    instructions="""
    You are a senior Python code reviewer at a tech company.
    
    YOUR ROLE:
    - Review Python code for bugs, style issues, and performance problems.
    - Rate code quality from 1-10.
    - Suggest specific improvements with corrected code snippets.
    
    YOUR RULES:
    - Always check for: type hints, docstrings, error handling, edge cases.
    - Be constructive, not harsh.
    - If the code is good, say so — don't invent problems.
    - Keep reviews concise (under 200 words).
    
    OUTPUT FORMAT:
    Score: X/10
    Issues: [list]
    Suggestions: [list with code]
    """,
)

result = await Runner.run(
    good_agent,
    """
Review this code:

def add(x, y):
    return x + y
""",
)

print(result.final_output)


**Score: 6/10**

**Issues:**
- No type hints (reduces readability and IDE support)
- Missing docstring explaining purpose
- No validation for incompatible types (e.g., `add("a", 1)` raises `TypeError`)
- No handling for `None` values

**Suggestions:**

```python
def add(x: int | float, y: int | float) -> int | float:
    """Add two numbers together.
    
    Args:
        x: First number
        y: Second number
    
    Returns:
        Sum of x and y
    
    Raises:
        TypeError: If x or y are not numeric types
    """
    if not isinstance(x, (int, float)) or not isinstance(y, (int, float)):
        raise TypeError(f"Both arguments must be numeric, got {type(x).__name__} and {type(y).__name__}")
    
    return x + y
```

This is a perfectly fine implementation for its simplicity — the improvements above are enhancements, not fixes for broken code.


In [10]:
from agents import Agent, Runner, ModelSettings

# Creative agent — high temperature
creative_agent = Agent(
    model=model,
    name="Poet",
    instructions="You write creative, imaginative descriptions. Be vivid and poetic.",
    model_settings=ModelSettings(
        temperature=0.9,  # High = more creative/random
        max_tokens=500,  # Limit output length
    ),
)

# Precise agent — low temperature
precise_agent = Agent(
    model=model,
    name="Fact Checker",
    instructions="You provide precise, factual answers. No speculation. Cite sources if possible.",
    model_settings=ModelSettings(
        temperature=0.1,  # Low = more deterministic/precise
        max_tokens=300,
    ),
)

# Same question, different agents
question = "Describe the Badshahi Mosque in Lahore."

creative_result = await Runner.run(creative_agent, question)
precise_result = await Runner.run(precise_agent, question)

print(" CREATIVE:\n", creative_result.final_output[:300])
print("\n PRECISE:\n", precise_result.final_output[:300])


 CREATIVE:
 # The Badshahi Mosque: A Verse in Marble and Sandstone

Imagine a vision carved from the dreams of emperors—rising from the heart of Lahore like a celestial fortress, where crimson sandstone and immaculate white marble collide in a duet of imperial ambition and spiritual devotion. This is the Badsha

 PRECISE:
 

# Badshahi Mosque

The Badshahi Mosque (also known as the "Emperor's Mosque") is a Mughal-era mosque located in Lahore, Pakistan. It stands as one of the largest mosques in the world and is a significant example of Mughal architecture.

## Construction and History
- Built between 1671 and 1673 dur


In [11]:
from pydantic import BaseModel, Field
from agents import Agent, Runner


# Define the output structure
class PodcastReview(BaseModel):
    title: str = Field(description="Podcast title")
    host: str = Field(description="Podcast host(s)")
    rating: float = Field(description="Rating from 1.0 to 10.0")
    genre: str = Field(
        description="Primary genre (e.g., AI Research, Machine Learning, AI News, AI Ethics)"
    )
    technical_level: str = Field(description="Beginner, Intermediate, or Advanced")
    summary: str = Field(description="One-sentence summary")
    recommendation: bool = Field(description="Would you recommend this?")


# Create agent with structured output
reviewer = Agent(
    model=model,
    name="AI Podcast Reviewer",
    instructions=(
        "You are an AI podcast critic. Analyze podcasts about artificial intelligence, "
        "machine learning, and related topics. Provide structured reviews and assess "
        "the technical depth and accessibility of each podcast."
    ),
    output_type=PodcastReview,
)

result = await Runner.run(reviewer, "Review the podcast 'Lex Fridman Podcast'")

# result.final_output is now a PodcastReview object, not a string!
review = result.final_output
print(f"Title:          {review.title}")
print(f"Host:           {review.host}")
print(f"Rating:         {review.rating}/10")
print(f"Genre:          {review.genre}")
print(f"Tech Level:     {review.technical_level}")
print(f"Summary:        {review.summary}")
print(f"Recommended:    {'Yes' if review.recommendation else 'No'}")


ModelBehaviorError: Invalid JSON when parsing 

# Podcast Review: Lex Fridman Podcast

## Overview
**Host:** Lex Fridman (researcher at MIT, previously at Google and Tesla Autopilot)
**Format:** Long-form interviews (typically 2-4 hours)
**Topics:** AI, deep learning, autonomous vehicles, consciousness, philosophy, science, technology

---

## Technical Depth Analysis

| Aspect | Assessment |
|--------|------------|
| **AI/ML Content** | ⭐⭐⭐⭐⭐ High depth with expert guests (researchers, engineers, founders) |
| **Mathematical Rigor** | ⭐⭐⭐ Moderate – often conceptual rather than technical |
| **Guest Variety** | ⭐⭐⭐⭐⭐ Diverse: researchers, philosophers, entrepreneurs, scientists |
| **Depth Consistency** | ⭐⭐⭐⭐ Generally high, varies by guest expertise |

**Strengths:**
- Explores fundamental questions about AI alignment, consciousness, AGI
- Deep technical interviews with leading researchers (Yoshua Bengio, DeepMind scientists)
- Covers emerging research trends thoroughly

**Limitatio

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from agents import Agent, Runner


class ProductClassification(BaseModel):
    category: Literal["electronics", "clothing", "food", "books", "other"] = Field(
        description="Product category"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgently does the customer need this?"
    )
    price_range: Literal["budget", "mid", "premium"] = Field(
        description="Estimated price range based on description"
    )
    search_query: str = Field(
        description="Optimized search query for the product catalog"
    )
    confidence: float = Field(description="Confidence score 0.0-1.0")


classifier = Agent(
    model="qwen2.5-coder:3b",
    name="Product Classifier",
    instructions="""
    You classify customer product requests for an e-commerce platform.
    Analyze the customer's natural language description and extract:
    - What category the product falls into
    - How urgent the request seems
    - Estimated price range
    - An optimized search query
    - Your confidence in the classification
    
    Be accurate. If you're unsure, set confidence low.
    """,
    output_type=ProductClassification,
    model_settings=ModelSettings(temperature=0.1),
)

# Test with different customer requests
requests = [
    "I need a laptop for my son's school, nothing too expensive",
    "Looking for a birthday gift, maybe a nice novel",
    "URGENT: Need a phone charger, mine just broke!",
]

for req in requests:
    result = await Runner.run(classifier, req)
    c = result.final_output
    print(f"\nRequest: '{req}'")
    print(f"   Category: {c.category} | Urgency: {c.urgency} | Price: {c.price_range}")
    print(f"   Search: '{c.search_query}' | Confidence: {c.confidence:.0%}")


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
from datetime import datetime
from agents import Agent, Runner


def dynamic_instructions(context, agent):
    """Instructions that change based on current state."""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M")
    return f"""
    You are a scheduling assistant.
    Current date/time: {current_time}
    
    Rules:
    - If it's morning (before 12pm), suggest productive tasks.
    - If it's afternoon, suggest meetings and collaborations.
    - If it's evening, suggest winding down activities.
    - Always be aware of the current time in your suggestions.
    """


scheduler = Agent(
    model=model,
    name="Smart Scheduler",
    instructions=dynamic_instructions,  # <-- Pass function, not string!
)

result = await Runner.run(scheduler, "What should I do right now?")
print(result.final_output)


In [ ]:
# ---

## Real-World Pattern: Customer Support Ticket Analyzer

# **Industry Problem:** Thousands of support tickets daily. Agent analyzes sentiment, priority, and routes them.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from agents import Agent, Runner, ModelSettings


class TicketAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative", "angry"] = Field(
        description="Customer sentiment"
    )
    priority: Literal["P0-critical", "P1-high", "P2-medium", "P3-low"] = Field(
        description="Ticket priority"
    )
    department: Literal["billing", "technical", "sales", "general"] = Field(
        description="Which department should handle this"
    )
    summary: str = Field(description="One-line summary for the dashboard")
    suggested_response: str = Field(description="Draft first response to customer")


ticket_analyzer = Agent(
    model=model,
    name="Ticket Analyzer",
    instructions="""
    You are an AI support ticket analyzer for a SaaS company.
    
    PRIORITY RULES:
    - P0-critical: Service down, data loss, security breach
    - P1-high: Feature broken, payment issues, angry customer
    - P2-medium: Bug reports, feature requests, how-to questions
    - P3-low: General feedback, suggestions, non-urgent inquiries
    
    DEPARTMENT ROUTING:
    - billing: Payment, subscription, invoice, refund
    - technical: Bugs, errors, API issues, integrations
    - sales: Pricing, enterprise plans, demos, upgrades
    - general: Everything else
    
    Write the suggested_response in a professional, empathetic tone.
    """,
    output_type=TicketAnalysis,
    model_settings=ModelSettings(temperature=0.2),
)

# Simulate incoming tickets
tickets = [
    "Your API has been returning 500 errors for 2 hours. Our production is DOWN. Fix this NOW!",
    "Hi, I was charged twice this month. Can you please look into it? Thanks.",
    "Love the product! Any chance you'll add dark mode soon?",
]

for ticket in tickets:
    result = await Runner.run(ticket_analyzer, ticket)
    t = result.final_output
    print(f"\n{'=' * 60}")
    print(f"Ticket: {ticket[:60]}...")
    print(f"Sentiment: {t.sentiment}")
    print(f"Priority:  {t.priority}")
    print(f"Route to:  {t.department}")
    print(f"Summary:   {t.summary}")
    print(f"Response:  {t.suggested_response[:150]}...")
